In [ ]:
import collections
import importlib

import matplotlib.pyplot as plt

import data.traces.dataset_reader
import pyine.data.utils.lmdb_io
import pyine.utils.code.blocks
import pyine.utils.code.execution
import pyine.utils.portability

In [ ]:
importlib.reload(pyine.data.utils.lmdb_io)
importlib.reload(data.traces.raw_dataset_reader)
importlib.reload(pyine.utils.code.blocks)
importlib.reload(pyine.utils.code.execution)
importlib.reload(pyine.utils.portability)

In [ ]:
# @@@@@@@@@@@ FIXME

parser = data.traces.raw_dataset_reader.DatasetReader(lmdb_path="../data/2025-03-31-v01.raw.lmdb")
target_difficulties = ["EASY"]
trace_count = len(parser)
print(f"{trace_count=}")
metadata = parser.get_metadata()
metadata_fields_too_big_to_print = ["key_map", "installed_packages"]
for metadata_key, metadata_value in metadata.items():
    if metadata_key in metadata_fields_too_big_to_print:  # too big to print, skip it
        continue
    print(f"{metadata_key}: {metadata_value}")

In [ ]:
difficulty_counts = collections.Counter()
for problem_idx, problem_data in enumerate(parser):
    difficulty_counts[problem_data["difficulty"]] += 1
print("Difficulty distribution:", dict(difficulty_counts))
plt.bar(difficulty_counts.keys(), difficulty_counts.values())
plt.xticks(rotation=45, ha="right")
plt.xlabel("Difficulty Level")
plt.ylabel("Frequency")
plt.title("Distribution of Problem Difficulties")
plt.show()
traced_executions = 0
traced_functions = 0
traced_blocks = 0
traced_lines = 0
traced_functions_per_solution = 0
for problem_idx, problem_data in enumerate(parser):
    if target_difficulties is not None and problem_data["difficulty"] not in target_difficulties:
        continue
    traced_executions += len(problem_data["trace_results"])
    for _, trace_res in problem_data["trace_results"].items():
        trace_res = pyine.utils.code.execution.TraceResult(**trace_res)
        traced_steps = [t for t in trace_res.traced_steps if t is not None]
        traced_source_lines: set[int] = set()
        for t in traced_steps:
            if t.trace_key.file == pyine.utils.code.execution.EXEC_TRACE_FILE_NAME:
                traced_source_lines.add(t.trace_key.line)
        traced_lines += len(traced_source_lines)
        for code_block_start_line, code_block_data in trace_res.code_blocks.items():
            if code_block_start_line not in traced_source_lines:
                continue
            traced_blocks += 1
            if code_block_data.type == pyine.utils.code.blocks.BlockType.FUNCTION:
                traced_functions += 1
print(f"{traced_executions=}")
print(f"{traced_functions=}")
print(f"{traced_blocks=}")
print(f"{traced_lines=}")